# Avance 2 — Modelo Relacional
**Proyecto Integrador M3 — TechCore / DataVision Analytics**

Este notebook toma el dataset limpio del Avance 1 (`ventasTransformed.csv`), identifica las
entidades del negocio y construye un **modelo relacional normalizado** con `pandas`, validando
la **integridad referencial** y exportando todas las tablas a `modeloVentas.xlsx` para alimentar
el dashboard de Power BI (Avance 3).

**Entidades:** Facturas, DetalleFacturas, Productos, Clientes, Sucursales, Ciudades, Vendedores.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('ventasTransformed.csv', encoding='utf-8-sig', low_memory=False)
print('Dataset limpio:', df.shape)
df.head(3)

Dataset limpio: (30000, 33)


,VentaID,FechaVenta,HoraVenta,SucursalNombre,CiudadSucursal,VendedorNombre,ClienteNombre,GeneroCliente,EdadCliente,EmailCliente,...,NombreProducto3,MarcaProducto3,CantidadProducto3,PrecioUnitarioProducto3,SubtotalProducto3,DescuentoVenta,TotalVenta,Año,Mes,Dia
0,240,07/12/2021,23:28:42,TechCore Medellín #2,Medellín,Claudia Sastre-Quintero,Pánfilo Salazar,M,25,pánfilo11@yahoo.com,...,NaN,NaN,NaN,NaN,NaN,0.0,5200000,2021,12,7
1,637,15/07/2017,23:02:30,TechCore Medellín #1,Medellín,Ana Sofía Llopis Blázquez,Crescencia Gutierrez Gimenez,F,18,crescencia98@hotmail.com,...,NaN,NaN,NaN,NaN,NaN,0.0,5200000,2017,7,15
2,921,16/06/2025,19:32:20,TechCore Bogotá #1,Bogotá,Poncio Gabriel Poza Acedo,Agustina Valcárcel Marco,F,25,agustina17@hotmail.com,...,NaN,NaN,NaN,NaN,NaN,0.0,5200000,2025,6,16


## 1. Ciudades
Dimensión simple con las 4 ciudades normalizadas en el Avance 1.

In [2]:
ciudades = (pd.DataFrame({'NombreCiudad': sorted(df.CiudadSucursal.unique())})
            .reset_index().rename(columns={'index':'CiudadID'}))
ciudades['CiudadID'] += 1
city2id = dict(zip(ciudades.NombreCiudad, ciudades.CiudadID))
ciudades

,CiudadID,NombreCiudad
0,1,Bogotá
1,2,Cali
2,3,Medellín
3,4,Pereira


## 2. Sucursales
Cada sucursal pertenece a una ciudad (FK `CiudadID`).

In [3]:
suc = (df[['SucursalNombre','CiudadSucursal']].drop_duplicates()
       .sort_values('SucursalNombre').reset_index(drop=True))
suc['SucursalID'] = suc.index + 1
suc['CiudadID'] = suc.CiudadSucursal.map(city2id)
sucursales = suc[['SucursalID','SucursalNombre','CiudadID']]
suc2id = dict(zip(suc.SucursalNombre, suc.SucursalID))
sucursales

,SucursalID,SucursalNombre,CiudadID
0,1,TechCore Bogotá #1,1
1,2,TechCore Bogotá #2,1
2,3,TechCore Cali,2
3,4,TechCore Medellín #1,3
4,5,TechCore Medellín #2,3
5,6,TechCore Pereira,4


## 3. Vendedores

In [4]:
vend = pd.DataFrame({'NombreVendedor': sorted(df.VendedorNombre.dropna().unique())})
vend['VendedorID'] = vend.index + 1
vendedores = vend[['VendedorID','NombreVendedor']]
vend2id = dict(zip(vend.NombreVendedor, vend.VendedorID))
print(len(vendedores), 'vendedores'); vendedores.head()

30 vendedores


,VendedorID,NombreVendedor
0,1,Almudena Adelaida Lledó Garrido
1,2,Amílcar Ortega-Alberto
2,3,Ana Sofía Llopis Blázquez
3,4,Anacleto Morera Segovia
4,5,Armida Azorin Plaza


## 4. Clientes
Se deduplica por `Email`, que actúa como identificador natural del cliente.

In [5]:
cl = (df[['EmailCliente','ClienteNombre','GeneroCliente','EdadCliente',
           'TelefonoCliente','DireccionCliente']]
      .sort_values('EmailCliente').drop_duplicates('EmailCliente').reset_index(drop=True))
cl['ClienteID'] = cl.index + 1
clientes = cl.rename(columns={'ClienteNombre':'Nombre','GeneroCliente':'Genero',
    'EdadCliente':'Edad','TelefonoCliente':'Telefono','EmailCliente':'Email',
    'DireccionCliente':'Direccion'})[['ClienteID','Nombre','Genero','Edad',
    'Telefono','Email','Direccion']]
email2id = dict(zip(cl.EmailCliente, cl.ClienteID))
print(len(clientes), 'clientes únicos'); clientes.head()

15816 clientes únicos


,ClienteID,Nombre,Genero,Edad,Telefono,Email,Direccion
0,1,Aarón Herrera Tello,M,37,+34 826 474 028,aarón11@hotmail.com,cll 2 #72-64
1,2,Aarón Valderrama Llanos,M,48,+34 821 36 87 11,aarón12@hotmail.com,cll 57 #71-66
2,3,Aarón Asenjo Reguera,M,30,+34714 538 644,aarón13@gmail.com,cll 16 #89-59
3,4,Aarón Asensio Martín,M,34,+34886 38 77 52,aarón14@yahoo.com,cll 93 #99-72
4,5,Aarón Marcos Crespo,M,45,+34814 370 071,aarón15@hotmail.com,cra 52 #80-19


## 5. Productos
El dataset crudo es **ancho** (hasta 3 productos por factura). Unimos los tres bloques y
deduplicamos por (Nombre, Marca). El precio se toma como la moda (precio de catálogo).

In [6]:
frames = []
for i in [1,2,3]:
    p = df[[f'NombreProducto{i}', f'MarcaProducto{i}', f'PrecioUnitarioProducto{i}']].copy()
    p.columns = ['Nombre','Marca','Precio']
    frames.append(p)
prod = pd.concat(frames).dropna(subset=['Nombre'])
productos = (prod.groupby(['Nombre','Marca'], as_index=False)['Precio']
             .agg(lambda s: s.mode().iat[0])
             .sort_values(['Marca','Nombre']).reset_index(drop=True))
productos['ProductoID'] = productos.index + 1
productos = productos[['ProductoID','Nombre','Marca','Precio']]
prod2id = dict(zip(zip(productos.Nombre, productos.Marca), productos.ProductoID))
print(len(productos), 'productos'); productos.head()

40 productos


,ProductoID,Nombre,Marca,Precio
0,1,Acer Aspire 5,Acer,2000000.0
1,2,Acer Nitro 5,Acer,4000000.0
2,3,Acer Predator Helios 300,Acer,5600000.0
3,4,Acer Swift 3,Acer,2600000.0
4,5,Apple MacBook Air M1,Apple,5500000.0


## 6. Facturas (cabecera)
Una fila por factura. FKs a Sucursal, Cliente y Vendedor.

In [7]:
facturas = pd.DataFrame({
    'FacturaID': df.VentaID.astype(str),
    'Fecha': df.FechaVenta, 'Hora': df.HoraVenta,
    'SucursalID': df.SucursalNombre.map(suc2id),
    'ClienteID':  df.EmailCliente.map(email2id),
    'VendedorID': df.VendedorNombre.map(vend2id),
    'MetodoPago': df.MetodoPago,
    'Descuento':  df.DescuentoVenta,
    'TotalVenta': df.TotalVenta,
    'Año': df['Año'], 'Mes': df['Mes'], 'Dia': df['Dia'],
})
print(facturas.shape); facturas.head(3)

(30000, 12)


,FacturaID,Fecha,Hora,SucursalID,ClienteID,VendedorID,MetodoPago,Descuento,TotalVenta,Año,Mes,Dia
0,240,07/12/2021,23:28:42,5,12555,9,Tarjeta Crédito,0.0,5200000,2021,12,7
1,637,15/07/2017,23:02:30,4,3631,3,Tarjeta Crédito,0.0,5200000,2017,7,15
2,921,16/06/2025,19:32:20,1,324,29,Tarjeta Crédito,0.0,5200000,2025,6,16


## 7. DetalleFacturas (líneas)
Hacemos *unpivot* de los 3 bloques de producto a formato largo: una fila por línea de venta.

In [8]:
rows = []
for i in [1,2,3]:
    sub = df[df[f'NombreProducto{i}'].notna()].copy()
    rows.append(pd.DataFrame({
        'FacturaID':  sub.VentaID.astype(str),
        'ProductoID': [prod2id.get((n,m)) for n,m in zip(sub[f'NombreProducto{i}'], sub[f'MarcaProducto{i}'])],
        'Cantidad':       sub[f'CantidadProducto{i}'].astype('Int64'),
        'PrecioUnitario': sub[f'PrecioUnitarioProducto{i}'].astype('Int64'),
        'Subtotal':       sub[f'SubtotalProducto{i}'].astype('Int64'),
    }))
detalle = pd.concat(rows).reset_index(drop=True)
detalle.insert(0, 'DetalleID', range(1, len(detalle)+1))
print(detalle.shape); detalle.head()

(60059, 6)


,DetalleID,FacturaID,ProductoID,Cantidad,PrecioUnitario,Subtotal
0,1,240,20,1,5200000,5200000
1,2,637,20,1,5200000,5200000
2,3,921,20,1,5200000,5200000
3,4,1096,20,1,5200000,5200000
4,5,1106,20,1,5200000,5200000


## 8. Validación de integridad referencial
No deben existir registros huérfanos: toda FK debe tener su PK correspondiente.

In [9]:
checks = {
 'Detalle.FacturaID sin Factura':  int((~detalle.FacturaID.isin(facturas.FacturaID)).sum()),
 'Detalle.ProductoID sin Producto':int(detalle.ProductoID.isna().sum()),
 'Factura.SucursalID sin Sucursal':int((~facturas.SucursalID.isin(sucursales.SucursalID)).sum()),
 'Factura.ClienteID sin Cliente':  int((~facturas.ClienteID.isin(clientes.ClienteID)).sum()),
 'Factura.VendedorID sin Vendedor':int((~facturas.VendedorID.isin(vendedores.VendedorID)).sum()),
}
for k,v in checks.items():
    print(('OK  ' if v==0 else 'ERR ')+f'{k}: {v}')
assert all(v==0 for v in checks.values()), 'Hay registros huérfanos'
print('\nIntegridad referencial validada ✔')

OK  Detalle.FacturaID sin Factura: 0
OK  Detalle.ProductoID sin Producto: 0
OK  Factura.SucursalID sin Sucursal: 0
OK  Factura.ClienteID sin Cliente: 0
OK  Factura.VendedorID sin Vendedor: 0

Integridad referencial validada ✔


## 9. Reportes exploratorios rápidos

In [10]:
det = detalle.merge(productos[['ProductoID','Nombre','Marca']], on='ProductoID')
print('VENTAS TOTALES POR MARCA')
print(det.groupby('Marca').Subtotal.sum().sort_values(ascending=False)
      .apply(lambda x: f'${x:,.0f}').to_string())

VENTAS TOTALES POR MARCA
Marca
Lenovo       $117,879,200,000
HP            $96,786,700,000
Dell          $86,722,000,000
Apple         $78,016,200,000
Asus          $33,179,600,000
Acer          $22,000,400,000
Samsung       $12,760,800,000
MSI            $8,525,200,000
Microsoft      $8,223,400,000
Razer          $3,936,400,000


In [11]:
print('TOP 10 PRODUCTOS MÁS VENDIDOS (unidades)')
print(det.groupby('Nombre').Cantidad.sum().sort_values(ascending=False).head(10).to_string())

TOP 10 PRODUCTOS MÁS VENDIDOS (unidades)
Nombre
HP Spectre x360              7726
Lenovo Legion 5 Pro          5861
Lenovo ThinkPad X1 Carbon    5814
Lenovo Yoga 7i               5052
HP Omen 16                   5011
Lenovo IdeaPad 5             4970
Dell XPS 13                  4526
Dell Latitude 7420           4507
HP Pavilion 15               4029
Dell Alienware m15           3658


## 10. Exportar `modeloVentas.xlsx`
Una hoja por tabla. Este archivo alimenta el dashboard de Power BI en el Avance 3.

In [12]:
with pd.ExcelWriter('modeloVentas.xlsx', engine='openpyxl') as xl:
    facturas.to_excel(xl,   sheet_name='Facturas',        index=False)
    detalle.to_excel(xl,    sheet_name='DetalleFacturas',  index=False)
    productos.to_excel(xl,  sheet_name='Productos',        index=False)
    clientes.to_excel(xl,   sheet_name='Clientes',         index=False)
    sucursales.to_excel(xl, sheet_name='Sucursales',       index=False)
    ciudades.to_excel(xl,   sheet_name='Ciudades',         index=False)
    vendedores.to_excel(xl, sheet_name='Vendedores',       index=False)
print('modeloVentas.xlsx exportado con 7 tablas.')

modeloVentas.xlsx exportado con 7 tablas.
